In [0]:
%sql
-- Nouveau catalog dédié à ce 3e projet, pour bien le séparer de BIXI et du streaming e-commerce
CREATE CATALOG IF NOT EXISTS tech_jobs_market;
USE CATALOG tech_jobs_market;

-- Avec Lakeflow Declarative Pipelines, on n'a pas besoin de créer les schémas bronze/silver/gold
-- à la main comme avant : le pipeline les créera lui-même au premier lancement.
-- On crée seulement un schéma "landing" pour y déposer les données brutes en amont.
CREATE SCHEMA IF NOT EXISTS landing;

-- Volume qui accueillera les fichiers CSV générés à l'étape 2
CREATE VOLUME IF NOT EXISTS landing.raw_files;

In [0]:
import random
import csv
from datetime import datetime, timedelta

# On fixe une graine aléatoire : les données générées seront toujours les mêmes
# à chaque exécution, ce qui facilite le débogage et la reproductibilité.
random.seed(42)

# Listes de valeurs réalistes pour construire des offres d'emploi cohérentes
titres = [
    "Data Analyst", "Data Engineer", "Data Scientist", "Développeur Python",
    "Ingénieur DevOps", "Architecte Cloud", "Business Intelligence Analyst",
    "Machine Learning Engineer", "Développeur Full Stack", "Administrateur BDD"
]
entreprises = [
    "TechNova", "DataCorp", "CloudWave", "InnoSoft", "ByteWorks",
    "Synapse Analytics", "PixelForge", "QuantumLeap", "NextGen Systems", "CoreLogic"
]
villes = ["Paris", "Lyon", "Toulouse", "Nantes", "Bordeaux", "Lille", "Marseille", "Remote"]
contrats = ["CDI", "CDD", "Freelance", "Alternance"]
niveaux = ["Junior", "Confirmé", "Senior"]

# Compétences groupées par thématique, pour piocher dedans de façon réaliste
# selon le poste (ex: un Data Engineer aura plus de chances d'avoir "Spark" qu'un Dev Full Stack)
competences_data = ["Python", "SQL", "Spark", "Databricks", "Airflow", "GCP", "Power BI", "dbt"]
competences_dev = ["JavaScript", "React", "Java", "Docker", "Kubernetes", "Git", "API REST"]

# On génère 5 fichiers CSV distincts, comme si 5 nouvelles journées de collecte
# d'offres avaient eu lieu — ça permettra de bien observer l'ingestion incrémentale
# (Auto Loader ne retraitera jamais un fichier déjà vu)
base_date = datetime(2026, 8, 1)

for jour in range(5):
    date_jour = base_date + timedelta(days=jour)
    chemin_fichier = f"/Volumes/tech_jobs_market/landing/raw_files/offres_{date_jour.strftime('%Y%m%d')}.csv"

    with open(chemin_fichier, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "job_id", "title", "company", "city", "contract_type",
            "remote", "experience_level", "required_skills",
            "salary_min", "salary_max", "posted_date", "source"
        ])

        # Entre 40 et 60 offres générées par jour
        for i in range(random.randint(40, 60)):
            job_id = f"{date_jour.strftime('%Y%m%d')}-{i:04d}"
            title = random.choice(titres)

            # On pioche dans le pool de compétences le plus adapté au métier
            pool = competences_data if "Data" in title or "BI" in title or "Machine" in title else competences_dev
            skills = random.sample(pool, k=random.randint(2, 5))

            niveau = random.choice(niveaux)
            # Le salaire varie selon le niveau d'expérience, pour que les agrégations Gold aient du sens plus tard
            base_salaire = {"Junior": 32000, "Confirmé": 42000, "Senior": 55000}[niveau]
            salary_min = base_salaire + random.randint(-2000, 2000)
            salary_max = salary_min + random.randint(5000, 12000)

            # Volontairement, on introduit ~3% de lignes "sales" (salaire manquant)
            # pour avoir de vraies données à filtrer avec les règles de qualité à l'étape Silver
            if random.random() < 0.03:
                salary_min, salary_max = "", ""

            writer.writerow([
                job_id, title, random.choice(entreprises), random.choice(villes),
                random.choice(contrats), random.choice(["Oui", "Non"]), niveau,
                ";".join(skills), salary_min, salary_max,
                date_jour.strftime("%Y-%m-%d"), "generated"
            ])

    print(f"Fichier généré : {chemin_fichier}")

In [0]:
display(dbutils.fs.ls("/Volumes/tech_jobs_market/landing/raw_files/"))